# The p+p hard tune in the A+A jet scheme

The hard tune was made with MATTER in its vacuum mode (`in_vac 1`): the shower stops at Q₀. The O+O and `pp_med`
runs use the A+A scheme (MATTER `in_vac 0` + LBT): outside the QGP (T ≤ T₀ = 0.16 GeV) MATTER showers down to
`QS` = 0.9 GeV, and QS also sets the Sudakov limit, the z range and the daughter virtualities. The O+O vacuum
showers therefore differ from the tuned ones. This notebook asks whether that matters against the p+p data
(tuning page, *Phase 4*, 2026-11-25).

**The two schemes, at the tune XML point** (`config/pp200_isr_rhictune.xml`: α_s 0.152, `PDF:pSet = 20`, C_v 0.25,
Q₀ 2, QS 0.9):

| Chain | Tune scheme | A+A scheme | Events per arm |
|---|---|---|---|
| vacuum chain (no strings, no hydro; π⁰ and η not decayed; p̂_T windows 5–30 GeV) | `--vacuum` (`vc_ref2`) | `--vacuum --aa-vacuum`: Brick at T = 0, Matter `in_vac 0`, no LBT (`vc_aa2`) | 294k (10 jobs) |
| full chain (iColored, strings, MUSIC where there is a medium; p̂_T windows 5–80 GeV) | `--vacuum-jet` (`aa_ref_h`) | `--matter-only`, Matter `T0 = hydro_Tc = 100`: no QGP anywhere (`aa_vac_h`) | 8k (4 jobs) |

Both arms of a chain use the same seeds. The vacuum chain gives the π⁰ (decays off) and the jets up to 26.6 GeV;
the full chain gives the jets over the whole fit range (its π⁰ decay, so it has no π⁰ here).

**Inputs.** The runs' observables in `$OO_RHIC_PROD/obs` (`observables.py`), and for the π⁰ ratio in wide p_T
ranges the event files of the vacuum chain (per job, about 2 s); commands in the README. `$OO_RHIC_PROD` defaults
to `~/prod/OO-RHIC_aascheme_recheck` here.

Model results, not vetted physics results; statistical errors only (the χ² include the data's errors).

In [ ]:
import sys as _sys
_sys.path.insert(0, "..")
import _paths  # noqa: E402,F401  (analysis/OO-RHIC and its test directories on sys.path; shared results/, refs/, data/ in ..)
import glob
import os
import sys
from pathlib import Path

os.environ.setdefault("OO_RHIC_PROD", str(Path.home() / "prod" / "OO-RHIC_aascheme_recheck"))
import matplotlib.pyplot as plt
import numpy as np

HERE = Path.cwd()
sys.path.insert(0, str(HERE))
import config as C          # noqa: E402
import observables as O     # noqa: E402

# palette: categorical slots in fixed order (validated reference palette), ink for data and ratios
INK, MUTED, GRID, BAND = "#1f1f1e", "#6b6a65", "#e4e3de", "#d6d5cf"
TUNE, AA = "#2a78d6", "#eb6834"
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.edgecolor": MUTED, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "lines.linewidth": 2, "lines.markersize": 6})

VC = ("vc_ref2", "vc_aa2")      # vacuum chain: tune scheme, A+A scheme
FC = ("aa_ref_h", "aa_vac_h")   # full chain
data = O.load_data()
OBS = {r: dict(np.load(C.obs_path(r))) for r in (*VC, *FC)}

# the four panels: data set, chain(s), the chi2 range per chain
PANELS = [("phenix_pi0", {"vc": (6, 20)}), ("star_pi0", {"vc": (6, 17)}),
          ("star_jets_eta0", {"vc": (8.2, 26.6), "fc": (8.2, 52)}),
          ("star_jets_eta1", {"vc": (8.2, 26.6), "fc": (8.2, 52)})]
CHAIN = {"vc": VC, "fc": FC}
CHAIN_NAME = {"vc": "vacuum chain", "fc": "full chain"}
for r in (*VC, *FC):
    js = [__import__("json").load(open(f)) for f in glob.glob(str(C.point_dir(r) / "*.json"))]
    print(f"{r:9s} {len(js):2d} jobs, {sum(j['events_written'] for j in js):6d} events, "
          f"skipped {sum(j['events_skipped'] for j in js)}")

## χ² of the two schemes

`tune.py`'s recipe: residuals in ln(model/data), the data's statistical and systematic errors per bin, the
normalisation uncertainty fully correlated, plus the model's statistical error. ⟨model/data⟩ is the geometric mean
over the range.

In [ ]:
def in_range(name, lo, hi):
    e = np.asarray(data[name]["edges"])
    return (e[:, 0] >= lo - 1e-9) & (e[:, 1] <= hi + 1e-9)


def chi2(run, name, lo, hi):
    m = in_range(name, lo, hi)
    d = np.asarray(data[name]["value"])[m]
    v, ev = O.combine(OBS[run], name, data)
    v, ev = v[m], ev[m]
    rel2 = (np.asarray(data[name]["stat"])[m] ** 2 + np.asarray(data[name]["sys"])[m] ** 2) / d ** 2
    res = np.log(v) - np.log(d)
    cov = np.diag(rel2 + (ev / v) ** 2) + (data[name].get("norm") or 0.0) ** 2
    return float(res @ np.linalg.solve(cov, res)), int(m.sum()), float(np.exp(np.mean(res)))


ROWS = [("phenix_pi0", 6, 20, "vc"), ("star_pi0", 6, 17, "vc"), ("star_pi0", 10, 17, "vc"),
        ("star_jets_eta0", 8.2, 26.6, "vc"), ("star_jets_eta1", 8.2, 26.6, "vc"),
        ("star_jets_eta0", 8.2, 52, "fc"), ("star_jets_eta1", 8.2, 52, "fc")]
CHI2 = {}
print(f"{'data set':15s} {'pT (GeV)':10s} {'chain':13s} {'tune scheme':>21s} {'A+A scheme':>21s}")
print(f"{'':40s} {'chi2 / n  <m/d>':>21s} {'chi2 / n  <m/d>':>21s}")
for name, lo, hi, ch in ROWS:
    (ca, n, ga), (cb, _, gb) = (chi2(r, name, lo, hi) for r in CHAIN[ch])
    CHI2[(name, ch, lo)] = (ca, cb, n)
    print(f"{name:15s} {f'{lo:g}-{hi:g}':10s} {CHAIN_NAME[ch]:13s} {ca:9.1f} / {n:2d}  {ga:.2f} "
          f"{cb:9.1f} / {n:2d}  {gb:.2f}")
print("normalisation uncertainty:", {k: data[k].get("norm") for k in ("phenix_pi0", "star_pi0", "star_jets_eta0")})

## A+A scheme / tune scheme

Per data bin, with the statistical errors of both arms (the arms share seeds but not events: the A+A shower draws
other random numbers). For the π⁰ also in wide p_T ranges, with the job-to-job spread as error (10 jobs per arm),
as in the table on the tuning page. Below p_T ≈ 6 GeV only events with p̂_T > 5 GeV contribute, so the π⁰ spectra
are incomplete there; the ratio still compares the two showers of the same hard events. Shaded: the χ² range.

In [ ]:
RANGES = [(2, 4), (4, 6), (6, 8), (8, 10), (10, 14), (14, 20)]
ACC = {"phenix_pi0": lambda y: np.abs(y) < 0.35, "star_pi0": lambda y: (y > 0) & (y < 1)}


def pi0_per_job(run):
    out = []
    for f in sorted(glob.glob(str(C.point_dir(run) / "*_hadrons_jet_frag.h5"))):
        fd = O.read_file(f)
        nev = np.bincount(fd["win"], minlength=len(fd["sigma"]))
        w = np.repeat((fd["sigma"] / np.maximum(nev, 1))[fd["win"]], fd["counts"])
        pt, y, _, _ = O._kin(fd["p"])
        pi0 = fd["pid"] == 111
        out.append({(k, r): np.sum(w[pi0 & acc(y) & (pt >= r[0]) & (pt < r[1])])
                    for k, acc in ACC.items() for r in RANGES + [(6, 20)]})
    return out


JOBS = {r: pi0_per_job(r) for r in VC}


def wide_ratio(name, r):
    a = np.array([j[(name, r)] for j in JOBS[VC[1]]])
    b = np.array([j[(name, r)] for j in JOBS[VC[0]]])
    q = a.mean() / b.mean()
    return q, q * np.hypot(a.std(ddof=1) / np.sqrt(len(a)) / a.mean(), b.std(ddof=1) / np.sqrt(len(b)) / b.mean())


print("pi0 yield A+A / tune scheme (vacuum chain, error from the job-to-job spread)")
for name in ACC:
    print(f"  {data[name]['label'].replace('$', '')}: " + ", ".join(
        f"{lo}-{hi}: {q:.3f}+-{e:.3f}" for (lo, hi) in RANGES + [(6, 20)] for q, e in [wide_ratio(name, (lo, hi))]))


def bin_ratio(run_a, run_b, name):
    va, ea = O.combine(OBS[run_a], name, data)
    vb, eb = O.combine(OBS[run_b], name, data)
    ok = (va > 0) & (vb > 0)
    q = np.where(ok, vb / np.where(ok, va, 1), np.nan)
    return q, q * np.hypot(ea / np.where(ok, va, 1), eb / np.where(ok, vb, 1))


fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
for ax, (name, rng) in zip(axes.flat, PANELS):
    e = np.asarray(data[name]["edges"])
    mid = e.mean(1)
    lo_plot = 2 if "pi0" in name else 8.2
    for k, (lo, hi) in rng.items():
        if k == "fc" or len(rng) == 1:
            ax.axvspan(lo, hi, color=GRID, alpha=0.5, lw=0, zorder=0)
    for k, (lo, hi) in rng.items():
        q, qe = bin_ratio(*CHAIN[k], name)
        m = (e[:, 0] >= lo_plot - 1e-9) & (e[:, 1] <= hi + 1e-9) & np.isfinite(q)
        off = -0.15 if k == "vc" and len(rng) > 1 else (0.15 if k == "fc" else 0)
        ax.errorbar(mid[m] + off, q[m], qe[m], fmt="o", color=INK, ms=4.5 if k == "vc" else 5.5,
                    mfc=INK if k == "vc" else "white", mew=1.4, elinewidth=1.2, capsize=0,
                    label=f"per bin, {CHAIN_NAME[k]}")
    if "pi0" in name:
        for i, r in enumerate(RANGES):
            q, qe = wide_ratio(name, r)
            ax.add_patch(plt.Rectangle((r[0], q - qe), r[1] - r[0], 2 * qe, color=AA, alpha=0.22, lw=0))
            ax.plot(r, [q, q], color=AA, lw=2.2, label="wide p$_T$ ranges, job spread" if i == 0 else None)
        ax.set_xlim(1.5, 20.5)
        ax.set_xlabel(r"$\pi^0$ $p_T$ (GeV)")
    else:
        ax.set_xlim(7.5, 53)
        ax.set_xlabel(r"jet $p_T$ (GeV)")
    ax.axhline(1, color=MUTED, lw=1, ls="--", zorder=1)
    ax.set_ylim(0.55, 1.65)
    ax.set_ylabel("A+A scheme / tune scheme")
    ax.set_title(data[name]["label"], loc="left", fontsize=10)
    ax.legend(fontsize=8, loc="upper left")
fig.suptitle("A+A-scheme vacuum shower / tune scheme (tune XML point, same seeds)", x=0.01, ha="left", fontsize=11)
plt.tight_layout()
plt.show()

## Model / data for both schemes

The band at 1 is the data's statistical and systematic error per bin; the normalisation uncertainty (fully
correlated) is quoted in each panel. The jets have both chains: filled markers the vacuum chain (to 26.6 GeV, its
windows end at p̂_T = 30 GeV), open markers the full chain (8k events). χ² / bins in the legend.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
for ax, (name, rng) in zip(axes.flat, PANELS):
    e = np.asarray(data[name]["edges"])
    d = np.asarray(data[name]["value"])
    rel = np.hypot(data[name]["stat"], data[name]["sys"]) / d
    full = max(hi for _, hi in rng.values())
    lo0 = min(lo for lo, _ in rng.values())
    mb = (e[:, 0] >= lo0 - 1e-9) & (e[:, 1] <= full + 1e-9)
    for i in np.where(mb)[0]:
        ax.fill_between(e[i], 1 - rel[i], 1 + rel[i], color=BAND, alpha=0.7, lw=0)
    ax.axhline(1, color=MUTED, lw=1, zorder=1)
    for k, (lo, hi) in rng.items():
        m = in_range(name, lo, hi)
        mid = e[m].mean(1)
        ca, cb, n = CHI2[(name, k, lo)]
        for j, (run, col, lab, c2) in enumerate(((CHAIN[k][0], TUNE, "tune scheme", ca),
                                                (CHAIN[k][1], AA, "A+A scheme", cb))):
            v, ev = O.combine(OBS[run], name, data)
            off = (-0.12 if j == 0 else 0.12) * (mid[1] - mid[0] if len(mid) > 1 else 1)
            ax.errorbar(mid + off + (0.25 if k == "fc" else 0), v[m] / d[m], ev[m] / d[m], fmt="o", color=col,
                        mfc=col if k == "vc" else "white", mew=1.4, ms=5, elinewidth=1.2, capsize=0,
                        label=f"{lab}, {CHAIN_NAME[k]}: χ² {c2:.1f} / {n}")
    norm = data[name].get("norm")
    ax.text(0.99, 0.03, f"norm. ±{norm * 100:.1f} %" if norm else "no norm. uncertainty given",
            transform=ax.transAxes, ha="right", fontsize=8, color=MUTED)
    ax.set_xlabel(("$\\pi^0$" if "pi0" in name else "jet") + " $p_T$ (GeV)")
    ax.set_ylabel("model / data")
    ax.set_ylim(0.4, 1.9)
    ax.set_title(data[name]["label"], loc="left", fontsize=10)
    ax.legend(fontsize=7.5, loc="upper left")
fig.suptitle("p+p 200 GeV against data, tune XML point (α_s 0.152, C_v 0.25, Q₀ 2)", x=0.01, ha="left", fontsize=11)
plt.tight_layout()
plt.show()

## Summary

- The A+A-scheme vacuum shower (down to QS instead of Q₀) changes the leading hadrons by a few percent at most:
  π⁰ 0.97–0.98 ± 0.02 at 2–4 GeV, 0.87–0.95 ± 0.05 at 4–6 GeV, consistent with 1 at 6–20 GeV.
- The jets do not change: 1.00 within 1–2 % per bin in the vacuum chain, and consistent with 1 in the full chain.
- Neither scheme is closer to the data: the χ² agree within the statistics for every data set.
- So the tune stands in the A+A scheme, no retune. For phase 3 the scheme mismatch between `oo_med` and
  `oo_vac`/`pp_ref` is a systematic of a few percent at most, which phase 3c (`pp_med` against `pp_ref`) measures;
  phase 4 (vacuum arms with `--aa-vacuum`) is optional.

Model results, not vetted physics results.